# HW6 : 총액 regression model 을 고르고, API 로 내보낸다

송민준 · 24013813

입력 : `sales_clean.csv` (HW5 단계 ④ 가 만든 파일). 이 노트북이 `sales_model.pkl` 을 만들고, `app.py` 가 그 파일을 읽어 `POST /api/sales/predict` 로 내보낸다.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# 한국어 윈도우에서 모델을 그림(HTML)으로 보여 주다 UnicodeDecodeError 가 나서 글자로 보여 준다
from sklearn import set_config
set_config(display="text")

## 단계 ① 읽고 나눈다

In [2]:
sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]          # feature : 수량, 단가
y = sales["total_sales"]                       # label : 총액 (숫자 → regression)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("전체 :", len(sales), " training :", len(X_train), " test :", len(X_test))

전체 : 1004  training : 803  test : 201


## 단계 ② model 셋의 test R²

In [3]:
# 새 feature qp = 수량 × 단가. training 과 test 둘 다에 더한다
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])

lin  = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression          test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp     test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit)   test R² :", round(tree.score(X_test, y_test), 3))

linear regression          test R² : 0.862
linear regression + qp     test R² : 1.0
decision tree (no limit)   test R² : 0.994


In [4]:
# linear 두 개가 찾은 규칙
print("linear      coef :", lin.coef_.round(1), " intercept :", round(lin.intercept_, 1))
print("linear + qp coef :", lin2.coef_.round(3), " intercept :", round(lin2.intercept_, 1))

linear      coef : [2.515409e+05 4.900000e+01]  intercept : -12387684.5
linear + qp coef : [0. 0. 1.]  intercept : -0.0


## 단계 ③ 답 : linear regression 이 0.862 에 그친 이유, qp 를 더하면 1.0 이 되는 이유

실제 규칙은 `총액 = 수량 × 단가` 로 두 열을 **곱한** 것이다. linear regression 의 모양은 `a × 수량 + b × 단가 + c` 처럼 열마다 숫자를 곱해 **더하는** 것뿐이라 곱셈을 표현할 수 없고, 그래서 직선으로 흉내 낸 0.862 에서 멈춘다.

qp 열을 더하면 곱셈이 이미 feature 안에 들어 있으므로 model 은 `총액 = 1 × qp` 만 찾으면 된다 (coef [0, 0, 1], 절편 0). model 은 그대로 두고 열 하나를 더해 0.862 → 1.0 이 되었다.

## 단계 ④ max_depth 별 training · test R²

In [5]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## 단계 ⑤ cross-validation 으로 max_depth 를 고른다

In [6]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.3f}  std {s.std():.3f}")

max_depth 3     mean 0.845  std 0.019
max_depth 5     mean 0.961  std 0.003
max_depth 8     mean 0.992  std 0.000


max_depth 12    mean 0.995  std 0.000


max_depth None  mean 0.995  std 0.001


**결정 : max_depth 8.** 평균이 8 에서 0.992, 12 에서 0.995 로 차이가 0.003 뿐인데 잎은 250 개 대 781 개로 세 배다. 점수가 거의 같으면 단순한 쪽을 고른다.

training 1.000 과 test 0.994 의 차이가 작은 것은 이 문제에 규칙 (총액 = 수량 × 단가) 이 실제로 있어서, 깊게 외워도 우연이 아니라 규칙을 외우게 되기 때문이다.

## 단계 ⑥ 고른 model 을 저장한다

In [7]:
import joblib

tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)

# 수량 40 · 단가 200,000 을 세 가지로
new = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
new2 = new.assign(qp=new["quantity"] * new["unit_price"])
print("linear + qp      ->", int(round(lin2.predict(new2)[0])))
print("tree (depth 8)   ->", int(round(tree8.predict(new)[0])))
print("quantity × price ->", 40 * 200000)

joblib.dump(lin2, "sales_model.pkl")           # linear regression + qp 를 서버에 올린다

linear + qp      -> 8000000
tree (depth 8)   -> 7362301
quantity × price -> 8000000


['sales_model.pkl']

In [8]:
# 저장한 파일을 다시 읽어 app.py 와 같은 방법으로 확인
loaded = joblib.load("sales_model.pkl")
x = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
x["qp"] = x["quantity"] * x["unit_price"]      # 학습 때 더한 feature 를 똑같이
print({"total_sales": int(round(loaded.predict(x)[0]))})

{'total_sales': 8000000}


## 단계 ⑦ 답 : 어느 model 인가, 왜

**linear regression + qp 를 올렸다.** 수량 40 · 단가 200,000 의 실제 총액은 40 × 200,000 = 8,000,000 원이다. linear + qp 는 8,000,000 으로 정확히 맞혔고, cross-validation 으로 고른 tree (depth 8) 는 7,362,301 로 약 64 만 원 (8 %) 틀렸다.

tree 는 잎마다 training 총액의 평균을 답하는 계단이라 곱셈을 근사할 뿐이고, linear + qp 는 실제 공식 `총액 = qp` 를 그대로 찾았다. 잎 250 개짜리 tree 보다 숫자 셋 (coef) 과 절편 하나인 model 이 더 단순하고 더 정확하다.